# 27 · Knowledge Graph RAG 与 GraphRAG

> 普通 RAG 只能“按段落找”。当问题需要跨文档、多跳、全局关系时（A 收购了 B，B 的 CEO 是谁），片段检索做不到，要用图。

**本文件覆盖知识点**：Knowledge Graph / Entity·Relation·Node·Edge / Entity Extraction / Relation Extraction / Graph Traversal / Graph Retrieval / Microsoft GraphRAG(Community Detection·Global/Local Search)

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. 普通 RAG vs 图 RAG

```text
普通 RAG:  Query → Vector → Documents（平铺片段，无法关联推理）
GraphRAG:  Query → 实体 → 图谱 → 关系/邻居 → Context（可多跳）
```

把文本先抽成实体（Entity，即节点）与关系（Relation，即边）：
```text
(阿里云) --收购了--> (百炼平台)     节点=实体，边=关系
(百炼平台) --提供--> (通义千问)
```
问“阿里云生态里谁提供大模型？”→ 从“阿里云”出发沿图走两跳即可回答。

In [ ]:
# 知识点·真调说明：实体/关系抽取 —— 真调一次 LLM，看一段文档如何变成图谱的“节点+边”
import json as _json
_doc = '通义千问是阿里云旗下百炼平台提供的大语言模型；百炼平台由阿里云收购并孵化，可对外提供模型服务。'
print('输入文档：', _doc)
print()
out = _llm_live(
    prompt='从下面文档中抽取全部“实体”和“实体间关系”，只输出一个 JSON 对象：\n'
           '{"entities": [{"name": "实体名", "type": "公司/产品/模型/技术等"}], '
           '"relations": [{"source": "头实体", "relation": "关系名", "target": "尾实体"}]}\n'
           '要求：只抽取文档里明说的事实，不要编造；entities 去重；每条 relation 都能读作“source relation target”。\n'
           '文档：' + _doc,
    system='你是知识图谱构建引擎。只输出 JSON 对象，禁止输出任何解释或代码块标记。',
    fallback='未配置 Key 的固定样例：\n'
             '{"entities": [{"name": "通义千问", "type": "产品"}, {"name": "阿里云", "type": "公司"}, '
             '{"name": "百炼平台", "type": "产品"}], '
             '"relations": [{"source": "百炼平台", "relation": "提供", "target": "通义千问"}, '
             '{"source": "阿里云", "relation": "收购并孵化", "target": "百炼平台"}]}',
    temperature=0.2,
)
if out is None:
    out = ('{"entities": [{"name": "通义千问", "type": "产品"}, {"name": "阿里云", "type": "公司"}, '
           '{"name": "百炼平台", "type": "产品"}], '
           '"relations": [{"source": "百炼平台", "relation": "提供", "target": "通义千问"}, '
           '{"source": "阿里云", "relation": "收购并孵化", "target": "百炼平台"}]}')
    print('（以上为固定样例；下面用样例走同一条解析）')
try:
    _g = _json.loads(out)
    print('json.loads 通过 ✅ 节点数=%d  关系数=%d' % (len(_g['entities']), len(_g['relations'])))
    for _e in _g['entities']:
        print('  [节点]', _e['name'], '/', _e['type'])
    for _r in _g['relations']:
        print('  [边]', _r['source'], '-', _r['relation'], '->', _r['target'])
except Exception as _e:
    print('未通过 json.loads：', _e)
print()
print('→ 一段文本就这样被“翻译”成 (节点,边)；GraphRAG 建图的第一步——Entity/Relation Extraction——正是靠这个环节自动完成，不再需要手工写边。')

In [ ]:
# 用 networkx 演示一个“微型知识图谱 + 图遍历”
try:
    import networkx as nx
except ImportError:
    print('未安装 networkx，运行 pip install networkx 后重试')
    raise SystemExit

g = nx.DiGraph()
# 实体与关系（生产：由 LLM 从文档抽取，即 Entity/Relation Extraction）
edges = [('阿里云','收购了','百炼平台'), ('百炼平台','提供','通义千问'),
         ('通义千问','用于','检索增强生成'), ('百炼平台','提供','qwen3-rerank')]
for s, r, t in edges:
    g.add_edge(s, t, relation=r)

def local_query(start):
    """本地(local)检索：从实体出发沿边取邻居"""
    for nbr in g.successors(start):
        rel = g[start][nbr]['relation']
        print(f'  {start} --{rel}--> {nbr}')
    for nbr in g.predecessors(start):
        rel = g[nbr][start]['relation']
        print(f'  {nbr} --{rel}--> {start}')

print('两跳推理: 阿里云 提供的模型包括 → 经百炼平台')
for n in ('百炼平台',):
    print(f'\n[{n}] 的邻居(1跳):'); local_query(n)

# 2跳: 从 阿里云 出发走两步
two_hop = list(nx.descendants(g, '阿里云'))
print('\n阿里云 可到达(2跳内):', two_hop)

## 2. Microsoft GraphRAG

微软 GraphRAG 思路（大规模落地版）：

```text
文档 → LLM 抽取 实体+关系 → 建 Entity Graph
     → Community Detection(社区发现): 把图切成若干社区
     → Hierarchical Summarization: 对社区逐层生成摘要(叶子→根)

查询时:
  Global Search  针对“全局/综述”类问题 → 用社区摘要
  Local Search   针对“具体实体”问题   → 走图遍历+相关片段
```

- Local 等价于带图的 RAG（实体锚定 + 邻居）；
- Global 是传统 RAG 做不到的“全库主题综述”，代价是离线建图成本高。



In [ ]:
# 知识点·真调说明：GraphRAG 层级摘要 —— 把“社区叶子摘要”聚合成可答综述题的全局摘要
_llm_live(
    prompt='以下是某企业知识库图谱做社区切分后得到的 3 篇叶子社区摘要，请把它们归纳成一段更高层的“全局主题综述”。\n'
           '[社区1] 星云机器人产品线：含客服机器人、工单机器人，均支持公有云与私有化两种部署。\n'
           '[社区2] 星云开放平台：提供对话 API 与自训练平台，开发者可训练自己的模型并接入机器人。\n'
           '[社区3] 星云安全能力：提供对话审计与私有化安全沙箱，满足金融、政务等合规要求。\n'
           '请输出一段能回答“星云产品的整体定位与核心能力是什么”的综述（不超过 120 字）。',
    system='你是 GraphRAG 的层级摘要器：把若干“社区叶子摘要”归纳成更高层的全局摘要，'
           '供 Global Search / 全局综述类问题使用。只输出综述正文，不要输出别的。',
    fallback='未配置 Key 的固定样例：\n'
             '星云以“智能客服机器人 + 开放平台”为核心：机器人产品线（客服/工单）支持公有云与私有化部署；'
             '开放平台提供对话 API 与自训练能力，支持开发者定制；并以对话审计、安全沙箱满足金融与政务合规。',
    temperature=0.2,
)
print('→ 传统 RAG 要遍历全库片段才敢做综述；GraphRAG 离线把图切成社区并逐层聚合摘要，'
      'Global Search 只需读这些摘要就能回答全局性问题——代价是建图成本高（适合关系推理场景）。')

## 小结

- 图 RAG 把文本变实体-关系图，支持多跳和全局问题；
- GraphRAG = 图 + 社区摘要，提供 Local/Global 双检索；
- 图构建成本高，适合需要关系推理的场景，与向量 RAG 并存使用。